# exp_e2_mpnet_multi — Semantic Graph Builder v2

**Phase 1a, embedder = `paraphrase-multilingual-mpnet-base-v2`, LLM = `deepseek-v32/latest` (API).**

Запускается из `exps/FINAL_EXPS/phase1a_embedders/exp_e2_mpnet_multi/` — все пути разрешаются автоматически от `clustering_1/`.

Перед запуском: `export YANDEX_CLOUD_API_KEY=...`.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import torch

In [4]:
import os, sys, logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# layout: clustering_1/exps/FINAL_EXPS/phase1a_embedders/exp_e2_mpnet_multi/
EXP_DIR   = Path().resolve()
REPO_ROOT = EXP_DIR.parents[3]                 # clustering_1/
LLM_V2    = REPO_ROOT / 'llm_v2'

# put repo root on sys.path so `import llm_v2` works
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
# guard: never expose llm_v2/ as flat path
if str(LLM_V2) in sys.path:
    sys.path.remove(str(LLM_V2))

from llm_v2.config_schema import load_config
config = load_config(EXP_DIR / 'config.yaml')

# expand ${YANDEX_CLOUD_API_KEY} etc. (no-op for local LLMs)
config.llm.api_key = os.path.expandvars(config.llm.api_key)
config.llm.base_url = os.path.expandvars(config.llm.base_url)
config.llm.folder = os.path.expandvars(config.llm.folder)

print('EXP_DIR  :', EXP_DIR)
print('REPO_ROOT:', REPO_ROOT)
print('LLM_V2   :', LLM_V2)
print()
print(config.model_dump_json(indent=2))

EXP_DIR  : /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1b_llms/exp_l05_mistral_small
REPO_ROOT: /home/platoon/graph/semantic-graph
LLM_V2   : /home/platoon/graph/semantic-graph/llm_v2

{
  "llm": {
    "provider": "local",
    "model_name": "mistralai/Mistral-Small-Instruct-2409",
    "max_new_tokens": 500,
    "temperature": 0.3,
    "device": "cuda",
    "load_in_8bit": false,
    "api_key": "",
    "base_url": "",
    "folder": "",
    "instructions": ""
  },
  "embedding": {
    "model_name": "intfloat/multilingual-e5-large",
    "device": "cuda"
  },
  "coreference": {
    "enabled": false,
    "prompt_file": "prompts/coreference_ru.txt",
    "context_sentences": 3,
    "window_sentences": 5
  },
  "extraction": {
    "prompt_file": "prompts/extraction_ru.txt",
    "chunk_size": 3,
    "overlap_size": 1
  },
  "normalization": {
    "enabled": true,
    "language": "ru"
  },
  "deduplication": {
    "enabled": true,
    "threshold": 0.92
  },
  "clustering": {
    "met

In [3]:
config.llm.api_key = ""

In [5]:
from llm_v2.models.llm_client import LLMClient
from llm_v2.models.embedder import Embedder

llm = LLMClient(config.llm)
embedder = Embedder(config.embedding)
print(f'LLM loaded: {config.llm.model_name}')
print(f'Embedder loaded: {config.embedding.model_name} (dim={embedder.dim})')

2026-05-07 16:45:16,017 [INFO] HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-Small-Instruct-2409/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 16:45:16,064 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mistralai/Mistral-Small-Instruct-2409/4600506f6b13c7ef89e61a54263f4c9bf483de30/config.json "HTTP/1.1 200 OK"
2026-05-07 16:45:16,240 [INFO] HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-Small-Instruct-2409/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 16:45:16,288 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mistralai/Mistral-Small-Instruct-2409/4600506f6b13c7ef89e61a54263f4c9bf483de30/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-07 16:45:16,470 [INFO] HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-Small-Instruct-2409/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 16:45:16,518 [INFO] HTTP Request: H

LLM loaded: mistralai/Mistral-Small-Instruct-2409
Embedder loaded: intfloat/multilingual-e5-large (dim=1024)


In [6]:
from llm_v2.utils.io import load_text

input_path = Path(config.paths.input_text)
if not input_path.is_absolute():
    input_path = (LLM_V2 / input_path).resolve()
text = load_text(input_path)
print(f'Input: {input_path}')
print(f'Length: {len(text)} chars')
print(text[:500])

Input: /home/platoon/graph/semantic-graph/benchmark/final_bench/formated_fragment2.md
Length: 15466 chars
# Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную классификацию на два класса. Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда.

Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$.

В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам 


## [0] Preprocessing

In [7]:
from llm_v2.stages.preprocessing import preprocess

sentences = preprocess(text, language=config.normalization.language)
for s in sentences:
    print(f'  [{s.id}] {s.text}')

  [0] # Линейная классификация

Теперь давайте поговорим про задачу классификации.
  [1] Для начала будем говорить про бинарную классификацию на два класса.
  [2] Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда.
  [3] Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$.
  [4] В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам будут нередко встречаться и метки $\{0,1\}$.
  [5] Мы хотим обучить линейную модель так, чтобы плоскость, которую она задаёт, как можно лучше отделяла объекты одного класса от объектов другого.
  [6] **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: положительный окажется с одной стороны от неё, а отрицательный — с другой.
  [7] Выборка, для которой это возможно, называется линейно разделимой.
  [8] Увы, в реальной жизни такое встречается кр

## [1] Coreference Resolution

In [8]:
from llm_v2.stages.coreference import resolve_coreferences

resolved_text, sentences = resolve_coreferences(
    sentences, llm, config.coreference, base_dir=LLM_V2
)
print('Resolved text:')
print(resolved_text)
print(f'\nSentences after coref: {len(sentences)}')

Resolved text:
# Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную классификацию на два класса. Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда. Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$. В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам будут нередко встречаться и метки $\{0,1\}$. Мы хотим обучить линейную модель так, чтобы плоскость, которую она задаёт, как можно лучше отделяла объекты одного класса от объектов другого. **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: положительный окажется с одной стороны от неё, а отрицательный — с другой. Выборка, для которой это возможно, называется линейно разделимой. Увы, в реальной жизни такое встречается крайне редко. Как обучить линейную модель

## [1.5] Chunking

In [9]:
from llm_v2.stages.chunking import build_chunks

chunks = build_chunks(sentences, config.extraction)
for c in chunks:
    print(f'  {c.id} (sents {c.sentence_ids}): {c.text[:80]}...')

  chunk_0 (sents [0, 1, 2]): # Линейная классификация

Теперь давайте поговорим про задачу классификации. Для...
  chunk_1 (sents [2, 3, 4]): Обобщить эту задачу до задачи классификации на $K$ классов не составит большого ...
  chunk_2 (sents [4, 5, 6]): В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам буду...
  chunk_3 (sents [6, 7, 8]): **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: пол...
  chunk_4 (sents [8, 9, 10]): Увы, в реальной жизни такое встречается крайне редко. Как обучить линейную модел...
  chunk_5 (sents [10, 11, 12]): $$

<details>
<summary>Почему бы не решать задачу классификации как задачу регре...
  chunk_6 (sents [12, 13, 14]): Во вторых, ошибкой будет считаться предсказание, например, $5$ вместо $1$, хотя ...
  chunk_7 (sents [14, 15, 16]): </details>

Сконструируем теперь функционал ошибки так, чтобы он вышеперечисленн...
  chunk_8 (sents [16, 17, 18]): $$

Домножим обе части на $y_i$ и немного упростим:

$

## [2] Triplet Extraction

In [10]:
from llm_v2.stages.extraction import extract_triplets

raw_triplets = extract_triplets(chunks, llm, config.extraction, base_dir=LLM_V2)
print(f'Extracted {len(raw_triplets)} raw triplets:')
for t in raw_triplets:
    print(f'  {t.subject} | {t.relation} | {t.object}  [{t.chunk_id}]')

Extracting triplets: 100%|██████████| 56/56 [09:10<00:00,  9.84s/it]

Extracted 577 raw triplets:
  линейная классификация | поговорим про | задача классификации  [chunk_0]
  линейная классификация | поговорим про | бинарная классификация  [chunk_0]
  бинарная классификация | на | два класса  [chunk_0]
  бинарная классификация | обобщить до | задача классификации  [chunk_0]
  задача классификации | обобщить до | $K$ классов  [chunk_0]
  задача | обобщается до | задача классификации на $K$ классов  [chunk_1]
  таргеты $y$ | кодируют | принадлежность к положительному или отрицательному классу  [chunk_1]
  таргеты $y$ | кодируют | множество $\{-1,1\}$  [chunk_1]
  векторы $x$ | принадлежат | $\mathbb{R}^D$  [chunk_1]
  классы | обозначаются | $\{-1,1\}$  [chunk_1]
  классы | встречаются | $\{0,1\}$  [chunk_1]
  параграф | договоримся обозначать | классы  [chunk_2]
  мы | хотим обучить | линейная модель  [chunk_2]
  линейная модель | задаёт | плоскость  [chunk_2]
  плоскость | разделит | классы  [chunk_2]
  плоскость | разделит положительный | отрицательный 

## [3] Normalization

In [11]:
from llm_v2.stages.normalization import normalize_triplets

norm_triplets = normalize_triplets(raw_triplets, config.normalization)
print(f'Normalized {len(norm_triplets)} triplets:')
for t in norm_triplets:
    print(f'  {t.norm_subject} | {t.norm_relation} | {t.norm_object}')

2026-05-07 16:55:29,998 [INFO] Loading dictionaries from /home/platoon/graph/graph_env/lib/python3.12/site-packages/pymorphy3_dicts_ru/data
2026-05-07 16:55:30,026 [INFO] format: 2.4, revision: 417150, updated: 2022-01-08T22:09:24.565962


Normalized 577 triplets:
  линейный классификация | поговорить про | задача классификация
  линейный классификация | поговорить про | бинарный классификация
  бинарный классификация | на | два класс
  бинарный классификация | обобщить до | задача классификация
  задача классификация | обобщить до | $k$ класс
  задача | обобщаться до | задача классификация на $k$ класс
  таргет $y$ | кодировать | принадлежность к положительный или отрицательный класс
  таргет $y$ | кодировать | множество $\{-1,1\}$
  вектор $x$ | принадлежать | $\mathbb{r}^d$
  класс | обозначаться | $\{-1,1\}$
  класс | встречаться | $\{0,1\}$
  параграф | договориться обозначать | класс
  мы | хотеть обучить | линейный модель
  линейный модель | задавать | плоскость
  плоскость | разделить | класс
  плоскость | разделить положительный | отрицательный
  компания apple | выпустить | iphone x
  iphone x | стать | самый дорогой смартфон в линейка apple
  компания apple | выпустить | iphone xr
  iphone xr | стать | самый п

## [4] Deduplication

In [12]:
from llm_v2.stages.deduplication import deduplicate_triplets

dedup_triplets = deduplicate_triplets(norm_triplets, embedder, config.deduplication)
print(f'After dedup: {len(norm_triplets)} -> {len(dedup_triplets)} triplets')
for t in dedup_triplets:
    print(f'  {t.norm_subject} | {t.norm_relation} | {t.norm_object}')

After dedup: 577 -> 216 triplets
  бинарный классификация | на | два класс
  бинарный классификация | обобщить до | задача классификация
  задача классификация | обобщить до | $k$ класс
  таргет $y$ | кодировать | принадлежность к положительный или отрицательный класс
  таргет $y$ | кодировать | множество $\{-1,1\}$
  вектор $x$ | принадлежать | $\mathbb{r}^d$
  класс | встречаться | $\{0,1\}$
  параграф | договориться обозначать | класс
  мы | хотеть обучить | линейный модель
  линейный модель | задавать | плоскость
  плоскость | разделить | класс
  компания apple | выпустить | iphone x
  iphone x | стать | самый дорогой смартфон в линейка apple
  iphone xr | стать | самый популярный смартфон в мир
  идеальный ситуация | найтись | плоскость
  класс | положительный | оказаться с один сторона от | плоскость
  выборка | для который это возможно | называться | линейно разделимый
  выборка | линейно разделимый | встречаться крайне редко
  встречаться | крайне редко | 
  итоговый предсказан

## [5] Graph Assembly (raw)

In [13]:
from llm_v2.stages.graph_assembly import assemble_graph

raw_graph = assemble_graph(dedup_triplets, chunks, text, config)
print(f'Raw graph: {len(raw_graph.nodes)} nodes, {len(raw_graph.edges)} edges')
print('\nNodes:')
for n in raw_graph.nodes:
    print(f'  {n.id}: {n.label} ({len(n.mentions)} mentions)')
print('\nEdges:')
for e in raw_graph.edges:
    print(f'  {e.id}: {e.source} --[{e.label}]--> {e.target} (w={e.weight})')

Raw graph: 297 nodes, 216 edges

Nodes:
  n0: бинарный классификация (2 mentions)
  n1: два класс (1 mentions)
  n2: задача классификация (3 mentions)
  n3: $k$ класс (1 mentions)
  n4: таргет $y$ (2 mentions)
  n5: принадлежность к положительный или отрицательный класс (1 mentions)
  n6: множество $\{-1,1\}$ (1 mentions)
  n7: вектор $x$ (1 mentions)
  n8: $\mathbb{r}^d$ (1 mentions)
  n9: класс (6 mentions)
  n10: $\{0,1\}$ (1 mentions)
  n11: параграф (1 mentions)
  n12: мы (19 mentions)
  n13: линейный модель (2 mentions)
  n14: плоскость (3 mentions)
  n15: компания apple (1 mentions)
  n16: iphone x (2 mentions)
  n17: самый дорогой смартфон в линейка apple (1 mentions)
  n18: iphone xr (1 mentions)
  n19: самый популярный смартфон в мир (1 mentions)
  n20: идеальный ситуация (1 mentions)
  n21: оказаться с один сторона от | плоскость (1 mentions)
  n22: выборка (2 mentions)
  n23: называться | линейно разделимый (1 mentions)
  n24: встречаться крайне редко (1 mentions)
  n25: вс

## [6] Clustering

In [14]:
from llm_v2.stages.clustering import cluster_graph, cluster_graph_multi, cluster_graph_all_methods
from llm_v2.utils.io import load_prompt

naming_prompt_path = Path(config.clustering.cluster_naming_prompt)
if not naming_prompt_path.is_absolute():
    naming_prompt_path = (LLM_V2 / naming_prompt_path).resolve()
naming_prompt = load_prompt(naming_prompt_path) if naming_prompt_path.exists() else None

if config.clustering.multi_method:
    multi = cluster_graph_all_methods(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )
    print('Multi-method clustering:')
    for method_name, mr in multi.methods.items():
        print(f'  {method_name}: {len(mr.param_labels)} variants')
        for lbl in mr.param_labels:
            g = mr.graphs[lbl]
            print(f'    {lbl}: {len(g.nodes)} nodes, {len(g.edges)} edges')
    agg = multi.methods['agglomerative']
    mid_label = agg.param_labels[len(agg.param_labels) // 2]
    clustered = agg.graphs[mid_label]
elif config.clustering.is_multi_threshold:
    multi = cluster_graph_multi(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )
    agg = multi.methods['agglomerative']
    print(f'Multi-threshold: {len(agg.param_labels)} levels')
    for lbl in agg.param_labels:
        g = agg.graphs[lbl]
        print(f'  t={lbl}: {len(g.nodes)} nodes, {len(g.edges)} edges')
    mid_label = agg.param_labels[len(agg.param_labels) // 2]
    clustered = agg.graphs[mid_label]
else:
    clustered = cluster_graph(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )

print(f'\nClustered graph: {len(clustered.nodes)} nodes, {len(clustered.edges)} edges')
print('\nClustered Nodes:')
for n in clustered.nodes:
    print(f'  {n.id}: {n.label} (members={n.members}, size={n.size})')
print('\nClustered Edges:')
for e in clustered.edges:
    print(f'  {e.id}: {e.source} --[{e.label}]--> {e.target} (size={e.size})')

Multi-method clustering:
  agglomerative: 10 variants
    0.250: 68 nodes, 69 edges
    0.322: 54 nodes, 52 edges
    0.394: 47 nodes, 45 edges
    0.467: 32 nodes, 39 edges
    0.539: 14 nodes, 20 edges
    0.611: 3 nodes, 1 edges
    0.683: 1 nodes, 0 edges
    0.756: 1 nodes, 0 edges
    0.828: 1 nodes, 0 edges
    0.900: 1 nodes, 0 edges
  kmeans: 4 variants
    k=10: 10 nodes, 21 edges
    k=25: 25 nodes, 38 edges
    k=40: 40 nodes, 46 edges
    k=55: 55 nodes, 56 edges
  hdbscan: 9 variants
    mcs=3,ms=1: 65 nodes, 63 edges
    mcs=3,ms=3: 75 nodes, 74 edges
    mcs=3,ms=5: 99 nodes, 83 edges
    mcs=5,ms=1: 72 nodes, 68 edges
    mcs=5,ms=3: 81 nodes, 78 edges
    mcs=5,ms=5: 99 nodes, 83 edges
    mcs=10,ms=1: 108 nodes, 80 edges
    mcs=10,ms=3: 114 nodes, 86 edges
    mcs=10,ms=5: 142 nodes, 107 edges

Clustered graph: 3 nodes, 1 edges

Clustered Nodes:
  c0: мы (members=['n0', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7', 'n8', 'n9', 'n10', 'n11', 'n12', 'n13', 'n14', 'n15', '

## Save outputs

In [15]:
from llm_v2.utils.io import save_json, save_text

out = EXP_DIR / config.paths.output_dir
out.mkdir(parents=True, exist_ok=True)

save_text(resolved_text, out / 'coreference_resolved.txt')
save_json(raw_graph.model_dump(), out / 'raw_graph.json')
save_json(clustered.model_dump(), out / 'clustered_graph.json')

if config.clustering.multi_method or config.clustering.is_multi_threshold:
    save_json(multi.model_dump(), out / 'multi_clustered_graph.json')
    method_counts = {m: len(r.param_labels) for m, r in multi.methods.items()}
    print(f'Saved multi_clustered_graph.json (methods: {method_counts})')

print(f'Saved to {out}/')

Saved multi_clustered_graph.json (methods: {'agglomerative': 10, 'kmeans': 4, 'hdbscan': 9})
Saved to /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1b_llms/exp_l05_mistral_small/output/


## Benchmark vs ground-truth graph

In [16]:
from llm_v2.benchmark import (
    evaluate_graph,
    evaluate_multi_graph,
    load_clustered_graph,
    print_metrics,
    print_multi_metrics,
    best_variant,
    show_node_alignments,
    show_edge_alignments,
    multi_metrics_to_dict,
)

# GT inputs (absolute, robust to CWD)
gt_graph_path = REPO_ROOT / 'benchmark' / 'final_bench' / 'graph_clustered.json'
gt_text_path  = REPO_ROOT / 'benchmark' / 'final_bench' / 'formated_fragment2.md'

gt_graph = load_clustered_graph(gt_graph_path)
gt_text  = gt_text_path.read_text(encoding='utf-8')

# embedding context: prefer the coreference-resolved text the pipeline saw
source_text = resolved_text if resolved_text else gt_text

print(f'GT  : {len(gt_graph.nodes)} nodes, {len(gt_graph.edges)} edges  ({gt_graph_path})')
print(f'Pred: {len(clustered.nodes)} nodes, {len(clustered.edges)} edges')
print(f'Context text: {len(source_text)} chars')

TAU_NODE = 0.6
TAU_EDGE = 0.6
BETA = 1.0
NODE_WEIGHT = 0.6
EDGE_WEIGHT = 0.4
NODE_WINDOW = 300
EDGE_WINDOW = 400
TOP_K = 10

GT  : 55 nodes, 51 edges  (/home/platoon/graph/semantic-graph/benchmark/final_bench/graph_clustered.json)
Pred: 3 nodes, 1 edges
Context text: 15418 chars


In [17]:
metrics = evaluate_graph(
    pred=clustered,
    gt=gt_graph,
    source_text=source_text,
    embedder=embedder,
    tau_node=TAU_NODE,
    tau_edge=TAU_EDGE,
    beta=BETA,
    node_weight=NODE_WEIGHT,
    edge_weight=EDGE_WEIGHT,
    node_window=NODE_WINDOW,
    edge_window=EDGE_WINDOW,
)

print_metrics(metrics)
metrics.summary()

Batches: 100%|██████████| 1/1 [00:00<00:00, 117.43it/s]


GraphScore = 0.6·NodeF1 + 0.4·EdgeF1  =  0.0688

Nodes (pred=3, gt=55, matched=3, tau=0.6, beta=1):
  TP(soft)  = 2.6410
  precision = 0.8803
  recall    = 0.0480
  F1        = 0.0911
Edges (pred=1, gt=51, matched=1, tau=0.6, beta=1):
  TP(soft)  = 0.9203
  precision = 0.9203
  recall    = 0.0180
  F1        = 0.0354


{'graph_score': 0.0687990402985631,
 'node_weight': 0.6,
 'edge_weight': 0.4,
 'nodes': {'precision': 0.8803266684214274,
  'recall': 0.048017818277532404,
  'f_beta': 0.09106827604359594,
  'beta': 1.0,
  'tau': 0.6,
  'tp': 2.6409800052642822,
  'pred_count': 3,
  'gt_count': 55,
  'matched_count': 3},
 'edges': {'precision': 0.9202748537063599,
  'recall': 0.018044604974634507,
  'f_beta': 0.03539518668101384,
  'beta': 1.0,
  'tau': 0.6,
  'tp': 0.9202748537063599,
  'pred_count': 1,
  'gt_count': 51,
  'matched_count': 1},
 'pred_structure': {'n_nodes': 3,
  'n_edges': 1,
  'density': 0.16666666666666666,
  'n_components': 2,
  'n_isolated': 1,
  'component_sizes': [2, 1],
  'component_size_min': 1,
  'component_size_max': 2,
  'component_size_mean': 1.5,
  'component_size_quantiles': {'q25': 1.25,
   'q50': 1.5,
   'q75': 1.75,
   'q90': 1.9}},
 'gt_structure': {'n_nodes': 55,
  'n_edges': 51,
  'density': 0.01717171717171717,
  'n_components': 6,
  'n_isolated': 0,
  'component_

In [18]:
show_node_alignments(clustered, gt_graph, metrics, top_k=TOP_K)

Matched node pairs: 3 / min(3, 55)=3

Top 3 matched (by quality q):
  [q=0.898]  'мы'  ↔  'число ошибок классификатора'
  [q=0.895]  'SVM'  ↔  'вектор'
  [q=0.848]  'гиперплоскость'  ↔  'гиперплоскость'

Unmatched GT nodes (52):
  'задача классификации'
  'бинарная классификация'
  'классификация на $K$ классов'
  'таргет $y$'
  'положительный класс'
  'отрицательный класс'
  'множество $\\{-1, 1\\}$'
  'признак $x_i$'
  'пространство $\\mathbb{R}^D$'
  'линейная модель'


In [19]:
show_edge_alignments(clustered, gt_graph, metrics, top_k=TOP_K)

Matched edge pairs: 1 / min(1, 51)=1

Top 1 matched (by quality q):
  [q=0.920]  мы —[выбирать]→ SVM
           ↔  вектор —[принадлежит]→ пространство $\mathbb{R}^D$

Unmatched GT edges (50):
  бинарная классификация —[является частным случаем]→ задача классификации
  бинарная классификация —[обобщается до]→ классификация на $K$ классов
  таргет $y$ —[кодирует принадлежность к]→ положительный класс
  таргет $y$ —[кодирует принадлежность к]→ отрицательный класс
  таргет $y$ —[принимает значения из]→ множество $\{-1, 1\}$
  признак $x_i$ —[является]→ вектор
  линейная модель —[параметризуется]→ веса $w$
  линейная модель —[задаёт]→ разделяющая плоскость
  разделяющая плоскость —[разделяет классы в]→ бинарная классификация
  выборка —[может обладать свойством]→ линейная разделимость


In [20]:
save_json(metrics.summary(), out / 'benchmark_metrics.json')
print(f'Saved benchmark_metrics.json to {out}/')

Saved benchmark_metrics.json to /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1b_llms/exp_l05_mistral_small/output/


## Benchmark — multi-method / multi-threshold sweep

In [ ]:
is_multi = config.clustering.multi_method or config.clustering.is_multi_threshold

if not is_multi:
    print('Skipped: multi-method / multi-threshold not enabled in config')
    multi_metrics = None
else:
    total = sum(len(mr.graphs) for mr in multi.methods.values())
    print(f'Evaluating {total} configurations...')
    multi_metrics = evaluate_multi_graph(
        multi=multi,
        gt=gt_graph,
        source_text=source_text,
        embedder=embedder,
        tau_node=TAU_NODE,
        tau_edge=TAU_EDGE,
        beta=BETA,
        node_weight=NODE_WEIGHT,
        edge_weight=EDGE_WEIGHT,
        node_window=NODE_WINDOW,
        edge_window=EDGE_WINDOW,
    )
    print(f'Done: {sum(len(v) for v in multi_metrics.values())} variants evaluated')

Evaluating 23 configurations...


Batches: 100%|██████████| 1/1 [00:00<00:00, 127.50it/s]


In [ ]:
if multi_metrics:
    print_multi_metrics(multi_metrics, sort_by='graph_score')

In [ ]:
if multi_metrics:
    method, param, best_m = best_variant(multi_metrics, by='graph_score')
    best_graph = multi.methods[method].graphs[param]
    print(f'Best variant: method={method}, param={param}')
    print(f'  graph: {len(best_graph.nodes)} nodes, {len(best_graph.edges)} edges')
    print()
    print_metrics(best_m)

In [ ]:
if multi_metrics:
    save_json(multi_metrics_to_dict(multi_metrics), out / 'benchmark_metrics_multi.json')
    print(f'Saved benchmark_metrics_multi.json to {out}/')